# Uncertainty-guided self-review

A LangGraph agent uses SAPLMA or UQ Head to score an archived draft,
reviews the most uncertain lines, and proposes a revision. Plain review
uses the same prompt and
generation budget with the uncertainty hints omitted.
[Agent pipeline diagram](../RESULTS.md#agent-pipeline)

Install the [root requirements](../requirements.txt) in the [project environment](README).
In VS Code, select the project interpreter at `.venv/Scripts/python.exe`.

For the Python checks, start Docker and run `docker pull python:3.11-slim`.
Set `check_python = False` to run the reference comparison alone.

In [1]:
import logging
from pathlib import Path
import sys

if Path("../src/traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import torch
from IPython.display import display

from datasets.benchmarks import DATASETS
from src.evaluation import AGENT_METHODS, TraceStore, run_benchmark

torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)
logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("src.methods").setLevel(logging.INFO)
cases_per_benchmark = 4
for name, estimator in AGENT_METHODS.items():
    print(f"{name}: {estimator.default_epochs} training epochs")


UQ Head: 10 training epochs
SAPLMA: 3 training epochs


Each method fits on that benchmark's original training split. Every
condition starts with the same held-out draft and gets one review round.
The agent sees the task, draft, and up to two ranked lines. The reference
solution and Python assertions are used by the evaluator.

Python cases are sampled from tasks whose canonical solution passes the archived assertions. The run uses a fixed seed.

SAPLMA trains for three epochs and UQ Head for ten in each approach.
Training logs show validation metrics before the agent generates revisions.

In [2]:
rows = []
review_training = []
for benchmark in DATASETS:
    print(f"Starting {benchmark}", flush=True)
    rows.extend(
        run_benchmark(
            store,
            benchmark,
            limit=cases_per_benchmark,
            check_python=True,
            training_history=review_training,
        )
    )
    print(f"Finished {benchmark}")
results = pd.DataFrame(rows)
for benchmark, group in results.groupby("benchmark", sort=False):
    print(benchmark, ", ".join(group["task"].unique()))

Starting mbpp


UncertaintyQuantificationHead epoch 1/10 train_loss=0.5838 validation_loss=0.4586 validation_token_pr_auc=0.2569 validation_token_roc_auc=0.7063
UncertaintyQuantificationHead epoch 2/10 train_loss=0.5369 validation_loss=0.4374 validation_token_pr_auc=0.2303 validation_token_roc_auc=0.6721
UncertaintyQuantificationHead epoch 3/10 train_loss=0.5298 validation_loss=0.4541 validation_token_pr_auc=0.2269 validation_token_roc_auc=0.6663
UncertaintyQuantificationHead epoch 4/10 train_loss=0.5296 validation_loss=0.4478 validation_token_pr_auc=0.2499 validation_token_roc_auc=0.6905
UncertaintyQuantificationHead epoch 5/10 train_loss=0.5364 validation_loss=0.4808 validation_token_pr_auc=0.2599 validation_token_roc_auc=0.7005
UncertaintyQuantificationHead epoch 6/10 train_loss=0.5360 validation_loss=0.4493 validation_token_pr_auc=0.3092 validation_token_roc_auc=0.7397
UncertaintyQuantificationHead epoch 7/10 train_loss=0.5233 validation_loss=0.4299 validation_token_pr_auc=0.3168 validation_token_

Finished mbpp
Starting humaneval



UncertaintyQuantificationHead epoch 1/10 train_loss=0.4612 validation_loss=0.5619 validation_token_pr_auc=0.2441 validation_token_roc_auc=0.4980
UncertaintyQuantificationHead epoch 2/10 train_loss=0.4210 validation_loss=0.5879 validation_token_pr_auc=0.2578 validation_token_roc_auc=0.4387
UncertaintyQuantificationHead epoch 3/10 train_loss=0.4051 validation_loss=0.6250 validation_token_pr_auc=0.2523 validation_token_roc_auc=0.4624
UncertaintyQuantificationHead epoch 4/10 train_loss=0.4043 validation_loss=0.5785 validation_token_pr_auc=0.2499 validation_token_roc_auc=0.4632
UncertaintyQuantificationHead epoch 5/10 train_loss=0.4088 validation_loss=0.5969 validation_token_pr_auc=0.2473 validation_token_roc_auc=0.4981
UncertaintyQuantificationHead epoch 6/10 train_loss=0.4154 validation_loss=0.6192 validation_token_pr_auc=0.2499 validation_token_roc_auc=0.5246
UncertaintyQuantificationHead epoch 7/10 train_loss=0.4145 validation_loss=0.6086 validation_token_pr_auc=0.2526 validation_token

Finished humaneval
Starting humaneval_java



UncertaintyQuantificationHead epoch 1/10 train_loss=0.5118 validation_loss=0.3554 validation_token_pr_auc=0.1524 validation_token_roc_auc=0.6423
UncertaintyQuantificationHead epoch 2/10 train_loss=0.4749 validation_loss=0.3786 validation_token_pr_auc=0.1604 validation_token_roc_auc=0.6561
UncertaintyQuantificationHead epoch 3/10 train_loss=0.5120 validation_loss=0.3501 validation_token_pr_auc=0.1943 validation_token_roc_auc=0.6822
UncertaintyQuantificationHead epoch 4/10 train_loss=0.4631 validation_loss=0.3859 validation_token_pr_auc=0.1862 validation_token_roc_auc=0.6753
UncertaintyQuantificationHead epoch 5/10 train_loss=0.4864 validation_loss=0.3468 validation_token_pr_auc=0.1724 validation_token_roc_auc=0.6493
UncertaintyQuantificationHead epoch 6/10 train_loss=0.4851 validation_loss=0.3758 validation_token_pr_auc=0.1716 validation_token_roc_auc=0.6477
UncertaintyQuantificationHead epoch 7/10 train_loss=0.4683 validation_loss=0.3558 validation_token_pr_auc=0.1865 validation_token

Finished humaneval_java
Starting defects4j



UncertaintyQuantificationHead epoch 1/10 train_loss=0.7362 validation_loss=0.6809 validation_token_pr_auc=0.6386 validation_token_roc_auc=0.7358
UncertaintyQuantificationHead epoch 2/10 train_loss=0.6931 validation_loss=0.6789 validation_token_pr_auc=0.6143 validation_token_roc_auc=0.7251
UncertaintyQuantificationHead epoch 3/10 train_loss=0.7162 validation_loss=0.6528 validation_token_pr_auc=0.6210 validation_token_roc_auc=0.7285
UncertaintyQuantificationHead epoch 4/10 train_loss=0.7090 validation_loss=0.6528 validation_token_pr_auc=0.6203 validation_token_roc_auc=0.7259
UncertaintyQuantificationHead epoch 5/10 train_loss=0.7174 validation_loss=0.6443 validation_token_pr_auc=0.6357 validation_token_roc_auc=0.7488
UncertaintyQuantificationHead epoch 6/10 train_loss=0.6519 validation_loss=0.7165 validation_token_pr_auc=0.6843 validation_token_roc_auc=0.7904
UncertaintyQuantificationHead epoch 7/10 train_loss=0.6734 validation_loss=0.6246 validation_token_pr_auc=0.6860 validation_token

Finished defects4j
mbpp 569, 391, 268, 723
humaneval 129, 127, 64, 115
humaneval_java 114, 36, 24, 139
defects4j Closure_52, Compress_30, Closure_35, JacksonDatabind_76


### Held-out results

Similarity compares code tokens with the archived canonical solution.
Exact matches use the same token representation. Python pass counts use
the archived assertions. Java results use the reference comparison.

In [3]:
plain = results.loc[results["method"] == "Plain review"]
baseline = plain.set_index(["benchmark", "task"])["similarity"]
results = results.join(baseline.rename("plain_similarity"), on=["benchmark", "task"])
results["delta_vs_plain"] = results["similarity"] - results["plain_similarity"]

grouped = results.groupby(["benchmark", "method"], sort=False)
summary = grouped.agg(
    tasks=("task", "size"),
    similarity=("similarity", "mean"),
    delta_vs_plain=("delta_vs_plain", "mean"),
    exact=("exact", "sum"),
    valid_replies=("accepted", "sum"),
)
summary["python_passes"] = grouped["passed"].sum(min_count=1).astype("Int64")
display(summary.round(4))

tasks  similarity  delta_vs_plain  exact  \
benchmark      method                                                     
mbpp           Archived draft      4      0.6357         -0.0913      0   
               Plain review        4      0.7269          0.0000      0   
               UQ Head             4      0.5893         -0.1377      0   
               SAPLMA              4      0.5893         -0.1377      0   
humaneval      Archived draft      4      0.4115         -0.1376      0   
               Plain review        4      0.5491          0.0000      0   
               UQ Head             4      0.5313         -0.0178      0   
               SAPLMA              4      0.5313         -0.0178      0   
humaneval_java Archived draft      4      0.5843          0.0525      0   
               Plain review        4      0.5318          0.0000      0   
               UQ Head             4      0.6399          0.1081      0   
               SAPLMA              4      0.6399          0.1081      0   
defects4j      Archived draft      4      0.2808          0.0355      0   
               Plain review        4      0.2453          0.0000      0   
               UQ Head             4      0.3148          0.0694      0   
               SAPLMA              4      0.3076          0.0623      0   

                               valid_replies  python_passes  
benchmark      method                                        
mbpp           Archived draft              4              0  
               Plain review                4              1  
               UQ Head                     4              2  
               SAPLMA                      4              2  
humaneval      Archived draft              4              0  
               Plain review                3              0  
               UQ Head                     3              0  
               SAPLMA                      3              0  
humaneval_java Archived draft              4           <NA>  
               Plain review                4           <NA>  
               UQ Head                     4           <NA>  
               SAPLMA                      4           <NA>  
defects4j      Archived draft              4           <NA>  
               Plain review                4           <NA>  
               UQ Head                     4           <NA>  
               SAPLMA                      4           <NA>

In [4]:
example = results.loc[results["method"] == "SAPLMA"].iloc[0]
print(f"{example['benchmark']} task {example['task']}")
print("Hints passed to the agent")
for line, excerpt in example["hints"]:
    print(f"{line}: {excerpt}")
print(example["review"])

mbpp task 569
Hints passed to the agent
3: return sorted(list1, key=lambda x: sorted(x))
1: def sort_sublists(list1):
```python
def sort_sublists(list1):
    return sorted(list1, key=lambda x: tuple(sorted(x)))
```

Review: The current draft is almost correct, but it needs a slight modification to ensure that the sorting is stable and the sublists are sorted in-place.

The issue with the current draft is that it sorts the sublists using the `sorted` function, which returns a new sorted list and leaves the original list unchanged. This can be inefficient if the sublists are large. To fix this, we can use the `sorted` function with the `key` argument set to a lambda function that returns a tuple of the sorted sublist. This ensures that the sorting is stable and the sublists are sorted in-place.

Alternatively, you could use the `sorted` function with the `key` argument set to a lambda function that returns the sorted sublist itself, like this:


In [5]:
print(*example)

mbpp 569 SAPLMA 0.9333333333333333 False True True [(3, 'return sorted(list1, key=lambda x: sorted(x))'), (1, 'def sort_sublists(list1):')] ```python
def sort_sublists(list1):
    return sorted(list1, key=lambda x: tuple(sorted(x)))
```

Review: The current draft is almost correct, but it needs a slight modification to ensure that the sorting is stable and the sublists are sorted in-place.

The issue with the current draft is that it sorts the sublists using the `sorted` function, which returns a new sorted list and leaves the original list unchanged. This can be inefficient if the sublists are large. To fix this, we can use the `sorted` function with the `key` argument set to a lambda function that returns a tuple of the sorted sublist. This ensures that the sorting is stable and the sublists are sorted in-place.

Alternatively, you could use the `sorted` function with the `key` argument set to a lambda function that returns the sorted sublist itself, like this: 0.9333333333333333 0.0

## Candidate selection

One model call proposes up to three distinct, complete code blocks.
Both estimators rank the same candidates. Each candidate is replayed
under the archived task prompt, and its score is the maximum predicted
defect score over code tokens. The agent keeps the lowest-scoring
candidate, with ties resolved in generation order. The draft is the
fallback when there are no usable candidates.

The two approaches start from the same held-out tasks. Line-guided
review allows 256 new tokens. Candidate selection allows up to 768
new tokens in one generation call. The evaluator scores only the
final answer returned by each approach.

In [6]:
from src.agent_loop import create_agent_candidates, extract_code

original = {"task": "Return the intended value.", "draft": "original"}
blocks = ["```python\nreturn " + str(i) + "\n```" for i in range(4)]
generated = "\n".join([blocks[0], blocks[0], *blocks[1:]])
scores = {"return 0": 0.8, "return 1": 0.2, "return 2": 0.2}
agent = create_agent_candidates(
    lambda prompt: generated,
    lambda reply: scores[extract_code(reply).strip()],
)
outcome = agent.invoke(original)
assert outcome["variants"] == blocks[:3]
assert outcome["uncertainties"] == [0.8, 0.2, 0.2]
assert outcome["selected"] == 1 and outcome["draft"] == blocks[1]

invalid = create_agent_candidates(lambda prompt: "```python\nreturn 0", lambda reply: float("nan"))
fallback = invalid.invoke(original)
assert fallback["selected"] is None and fallback["draft"] == original["draft"]
rejected = create_agent_candidates(lambda prompt: blocks[0], lambda reply: float("nan"))
assert rejected.invoke(original)["draft"] == original["draft"]
print("Candidate selection smoke checks passed.")

Candidate selection smoke checks passed.


In [7]:
from src.evaluation import run_benchmark

candidate_rows = []
for benchmark in DATASETS:
    print(f"Starting {benchmark}", flush=True)
    candidate_rows.extend(
        run_benchmark(
            store,
            benchmark,
            limit=cases_per_benchmark,
            check_python=True,
            max_candidates=3,
        )
    )
    print(f"Finished candidate selection on {benchmark}")
candidate_results = pd.DataFrame(candidate_rows)

Starting mbpp


UncertaintyQuantificationHead epoch 1/10 train_loss=0.5838 validation_loss=0.4586 validation_token_pr_auc=0.2569 validation_token_roc_auc=0.7063
UncertaintyQuantificationHead epoch 2/10 train_loss=0.5369 validation_loss=0.4374 validation_token_pr_auc=0.2303 validation_token_roc_auc=0.6721
UncertaintyQuantificationHead epoch 3/10 train_loss=0.5298 validation_loss=0.4541 validation_token_pr_auc=0.2269 validation_token_roc_auc=0.6663
UncertaintyQuantificationHead epoch 4/10 train_loss=0.5296 validation_loss=0.4478 validation_token_pr_auc=0.2499 validation_token_roc_auc=0.6905
UncertaintyQuantificationHead epoch 5/10 train_loss=0.5364 validation_loss=0.4808 validation_token_pr_auc=0.2599 validation_token_roc_auc=0.7005
UncertaintyQuantificationHead epoch 6/10 train_loss=0.5360 validation_loss=0.4493 validation_token_pr_auc=0.3092 validation_token_roc_auc=0.7397
UncertaintyQuantificationHead epoch 7/10 train_loss=0.5233 validation_loss=0.4299 validation_token_pr_auc=0.3168 validation_token_

Finished candidate selection on mbpp
Starting humaneval



UncertaintyQuantificationHead epoch 1/10 train_loss=0.4612 validation_loss=0.5619 validation_token_pr_auc=0.2441 validation_token_roc_auc=0.4980
UncertaintyQuantificationHead epoch 2/10 train_loss=0.4210 validation_loss=0.5879 validation_token_pr_auc=0.2578 validation_token_roc_auc=0.4387
UncertaintyQuantificationHead epoch 3/10 train_loss=0.4051 validation_loss=0.6250 validation_token_pr_auc=0.2523 validation_token_roc_auc=0.4624
UncertaintyQuantificationHead epoch 4/10 train_loss=0.4043 validation_loss=0.5785 validation_token_pr_auc=0.2499 validation_token_roc_auc=0.4632
UncertaintyQuantificationHead epoch 5/10 train_loss=0.4088 validation_loss=0.5969 validation_token_pr_auc=0.2473 validation_token_roc_auc=0.4981
UncertaintyQuantificationHead epoch 6/10 train_loss=0.4154 validation_loss=0.6192 validation_token_pr_auc=0.2499 validation_token_roc_auc=0.5246
UncertaintyQuantificationHead epoch 7/10 train_loss=0.4145 validation_loss=0.6086 validation_token_pr_auc=0.2526 validation_token

Finished candidate selection on humaneval
Starting humaneval_java



UncertaintyQuantificationHead epoch 1/10 train_loss=0.5118 validation_loss=0.3554 validation_token_pr_auc=0.1524 validation_token_roc_auc=0.6423
UncertaintyQuantificationHead epoch 2/10 train_loss=0.4749 validation_loss=0.3786 validation_token_pr_auc=0.1604 validation_token_roc_auc=0.6561
UncertaintyQuantificationHead epoch 3/10 train_loss=0.5120 validation_loss=0.3501 validation_token_pr_auc=0.1943 validation_token_roc_auc=0.6822
UncertaintyQuantificationHead epoch 4/10 train_loss=0.4631 validation_loss=0.3859 validation_token_pr_auc=0.1862 validation_token_roc_auc=0.6753
UncertaintyQuantificationHead epoch 5/10 train_loss=0.4864 validation_loss=0.3468 validation_token_pr_auc=0.1724 validation_token_roc_auc=0.6493
UncertaintyQuantificationHead epoch 6/10 train_loss=0.4851 validation_loss=0.3758 validation_token_pr_auc=0.1716 validation_token_roc_auc=0.6477
UncertaintyQuantificationHead epoch 7/10 train_loss=0.4683 validation_loss=0.3558 validation_token_pr_auc=0.1865 validation_token

Finished candidate selection on humaneval_java
Starting defects4j



UncertaintyQuantificationHead epoch 1/10 train_loss=0.7362 validation_loss=0.6809 validation_token_pr_auc=0.6386 validation_token_roc_auc=0.7358
UncertaintyQuantificationHead epoch 2/10 train_loss=0.6931 validation_loss=0.6789 validation_token_pr_auc=0.6143 validation_token_roc_auc=0.7251
UncertaintyQuantificationHead epoch 3/10 train_loss=0.7162 validation_loss=0.6528 validation_token_pr_auc=0.6210 validation_token_roc_auc=0.7285
UncertaintyQuantificationHead epoch 4/10 train_loss=0.7090 validation_loss=0.6528 validation_token_pr_auc=0.6203 validation_token_roc_auc=0.7259
UncertaintyQuantificationHead epoch 5/10 train_loss=0.7174 validation_loss=0.6443 validation_token_pr_auc=0.6357 validation_token_roc_auc=0.7488
UncertaintyQuantificationHead epoch 6/10 train_loss=0.6519 validation_loss=0.7165 validation_token_pr_auc=0.6843 validation_token_roc_auc=0.7904
UncertaintyQuantificationHead epoch 7/10 train_loss=0.6734 validation_loss=0.6246 validation_token_pr_auc=0.6860 validation_token

Finished candidate selection on defects4j


In [8]:
review = results.loc[
    results["method"].isin(candidate_results["method"].unique())
]
keys = ["benchmark", "task", "method"]
candidate_index = candidate_results.set_index(keys).index
review_index = review.set_index(keys).index
assert candidate_index.is_unique and review_index.is_unique
assert candidate_index.symmetric_difference(review_index).empty

paired = pd.concat([
    review.assign(approach="Line-guided review"),
    candidate_results.assign(approach="Candidate selection"),
], ignore_index=True)
groups = paired.groupby(["benchmark", "method", "approach"], sort=False)
comparison = groups.agg(
    tasks=("task", "size"),
    similarity=("similarity", "mean"),
    exact=("exact", "sum"),
    valid_replies=("accepted", "sum"),
    mean_candidates=("variants", "mean"),
)
comparison["python_passes"] = groups["passed"].sum(min_count=1).astype("Int64")
display(comparison.round(4))

tasks  similarity  exact  \
benchmark      method  approach                                        
mbpp           UQ Head Line-guided review       4      0.5893      0   
               SAPLMA  Line-guided review       4      0.5893      0   
humaneval      UQ Head Line-guided review       4      0.5313      0   
               SAPLMA  Line-guided review       4      0.5313      0   
humaneval_java UQ Head Line-guided review       4      0.6399      0   
               SAPLMA  Line-guided review       4      0.6399      0   
defects4j      UQ Head Line-guided review       4      0.3148      0   
               SAPLMA  Line-guided review       4      0.3076      0   
mbpp           UQ Head Candidate selection      4      0.5468      0   
               SAPLMA  Candidate selection      4      0.7090      0   
humaneval      UQ Head Candidate selection      4      0.5472      0   
               SAPLMA  Candidate selection      4      0.5845      0   
humaneval_java UQ Head Candidate selection      4      0.4096      0   
               SAPLMA  Candidate selection      4      0.4923      0   
defects4j      UQ Head Candidate selection      4      0.1587      0   
               SAPLMA  Candidate selection      4      0.1538      0   

                                            valid_replies  mean_candidates  \
benchmark      method  approach                                              
mbpp           UQ Head Line-guided review               4              NaN   
               SAPLMA  Line-guided review               4              NaN   
humaneval      UQ Head Line-guided review               3              NaN   
               SAPLMA  Line-guided review               3              NaN   
humaneval_java UQ Head Line-guided review               4              NaN   
               SAPLMA  Line-guided review               4              NaN   
defects4j      UQ Head Line-guided review               4              NaN   
               SAPLMA  Line-guided review               4              NaN   
mbpp           UQ Head Candidate selection              4             3.00   
               SAPLMA  Candidate selection              4             3.00   
humaneval      UQ Head Candidate selection              4             2.75   
               SAPLMA  Candidate selection              4             2.75   
humaneval_java UQ Head Candidate selection              4             2.00   
               SAPLMA  Candidate selection              4             2.00   
defects4j      UQ Head Candidate selection              4             1.50   
               SAPLMA  Candidate selection              4             1.50   

                                            python_passes  
benchmark      method  approach                            
mbpp           UQ Head Line-guided review               2  
               SAPLMA  Line-guided review               2  
humaneval      UQ Head Line-guided review               0  
               SAPLMA  Line-guided review               0  
humaneval_java UQ Head Line-guided review            <NA>  
               SAPLMA  Line-guided review            <NA>  
defects4j      UQ Head Line-guided review            <NA>  
               SAPLMA  Line-guided review            <NA>  
mbpp           UQ Head Candidate selection              1  
               SAPLMA  Candidate selection              2  
humaneval      UQ Head Candidate selection              0  
               SAPLMA  Candidate selection              0  
humaneval_java UQ Head Candidate selection           <NA>  
               SAPLMA  Candidate selection           <NA>  
defects4j      UQ Head Candidate selection           <NA>  
               SAPLMA  Candidate selection           <NA>

In [9]:
first_task = candidate_results.iloc[0]["task"]
first_benchmark = candidate_results.iloc[0]["benchmark"]
scores = candidate_results.loc[
    (candidate_results["benchmark"] == first_benchmark)
    & (candidate_results["task"] == first_task),
    ["method", "uncertainties", "selected_candidate"],
].copy()
scores["selected_candidate"] = (scores["selected_candidate"] + 1).astype("Int64")
print(f"Candidate scoring example for {first_benchmark} task {first_task}")
display(scores.set_index("method"))

Candidate scoring example for mbpp task 569


,uncertainties,selected_candidate
method,,
UQ Head,"[0.6511372327804565, 0.6496291756629944, 0.634...",3
SAPLMA,"[0.07092644274234772, 0.08097737282514572, 0.0...",1


### Estimator training history

The two review approaches independently fit the same estimators and use the same training and validation tasks. This table displays the line-guided run's complete history.

In [10]:
with pd.option_context("display.max_rows", None):
    training = pd.DataFrame(review_training)
    display(training.set_index(["benchmark", "method", "epoch"]).round(4))

train_loss  validation_loss  \
benchmark      method  epoch                                
mbpp           UQ Head 1          0.5838           0.4586   
                       2          0.5369           0.4374   
                       3          0.5298           0.4541   
                       4          0.5296           0.4478   
                       5          0.5364           0.4808   
                       6          0.5360           0.4493   
                       7          0.5233           0.4299   
                       8          0.5173           0.4141   
                       9          0.4892           0.3915   
                       10         0.4711           0.4931   
               SAPLMA  1          0.4012           0.3106   
                       2          0.2199           0.2734   
                       3          0.1527           0.3017   
humaneval      UQ Head 1          0.4612           0.5619   
                       2          0.4210           0.5879   
                       3          0.4051           0.6250   
                       4          0.4043           0.5785   
                       5          0.4088           0.5969   
                       6          0.4154           0.6192   
                       7          0.4145           0.6086   
                       8          0.4070           0.6127   
                       9          0.4080           0.5765   
                       10         0.4162           0.6070   
               SAPLMA  1          0.3904           0.6342   
                       2          0.2759           0.9459   
                       3          0.2115           1.1334   
humaneval_java UQ Head 1          0.5118           0.3554   
                       2          0.4749           0.3786   
                       3          0.5120           0.3501   
                       4          0.4631           0.3859   
                       5          0.4864           0.3468   
                       6          0.4851           0.3758   
                       7          0.4683           0.3558   
                       8          0.4658           0.3593   
                       9          0.4679           0.3555   
                       10         0.4667           0.3642   
               SAPLMA  1          0.4791           0.3342   
                       2          0.3725           0.3481   
                       3          0.3182           0.3781   
defects4j      UQ Head 1          0.7362           0.6809   
                       2          0.6931           0.6789   
                       3          0.7162           0.6528   
                       4          0.7090           0.6528   
                       5          0.7174           0.6443   
                       6          0.6519           0.7165   
                       7          0.6734           0.6246   
                       8          0.5328           0.4787   
                       9          0.4403           0.5380   
                       10         0.5623           0.7199   
               SAPLMA  1          0.4373           0.5356   
                       2          0.2969           0.5710   
                       3          0.2396           0.6552   

                              validation_token_pr_auc  \
benchmark      method  epoch                            
mbpp           UQ Head 1                       0.2569   
                       2                       0.2303   
                       3                       0.2269   
                       4                       0.2499   
                       5                       0.2599   
                       6                       0.3092   
                       7                       0.3168   
                       8                       0.3327   
                       9                       0.3718   
                       10                      0.4546   
               SAPLMA  1                       0.56